# BIOM 200: Microbiome analysis with QIIME 2

Analyze a 16S rRNA amplicon dataset from Qiita, from feature table to differential abundance. Run the cells in order; edit only the cells marked **EDIT**.

## 1. Why this matters

The microbes living on and in us are shaped by, and in turn shape, our immune system, our metabolism, and our susceptibility to pathogens. 16S rRNA amplicon sequencing is the most widely used way to ask which bacteria are present in a sample and in what proportions: it is cheap, fast enough to go from swab to data in a day, and supported by mature, peer-reviewed software. It also has well-known limits. It measures relative rather than absolute abundance, it rarely resolves species, and small studies are easily misled by contamination, sequencing depth, and multiple testing. This notebook walks through a standard analysis with those limits in view: visualize composition, account for uneven sequencing depth, compare diversity within and between samples, and test which taxa differ between groups.

## 2. Set up QIIME 2

This notebook uses the **QIIME 2 2026.7** distribution. Choose one option.

### Option A (recommended): GitHub Codespaces

Nothing to install. On the repository page, click **Code → Codespaces → Create codespace on main**. The first start takes a few minutes; QIIME 2 and this notebook open in your browser. When asked for a kernel, choose **QIIME 2 2026.7**. Stop the codespace when you finish (**Codespaces** menu → **Stop Current Codespace**) to save your free monthly hours.

### Option B: install on your own computer

Full instructions: [QIIME 2 installation guide](https://library.qiime2.org/quickstart/amplicon).

1. Install [Miniconda](https://www.anaconda.com/docs/getting-started/miniconda/install). **Windows:** first [install WSL](https://learn.microsoft.com/en-us/windows/wsl/install), then install Miniconda *inside* WSL and follow the Linux instructions.
2. Create the environment for your system (takes 10–30 minutes):

**Linux or Windows (WSL)**
```bash
conda env create --name rachis-qiime2-2026.7 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.7/qiime2/released/rachis-qiime2-linux-64-conda.yml
```

**macOS, Apple Silicon (M1 and later)**: runs under Rosetta 2
```bash
CONDA_SUBDIR=osx-64 conda env create --name rachis-qiime2-2026.7 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.7/qiime2/released/rachis-qiime2-osx-64-conda.yml
conda activate rachis-qiime2-2026.7
conda config --env --set subdir osx-64
```

**macOS, Intel**
```bash
conda env create --name rachis-qiime2-2026.7 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.7/qiime2/released/rachis-qiime2-osx-64-conda.yml
```

3. Add Jupyter and start it from the folder containing this notebook:
```bash
conda activate rachis-qiime2-2026.7
conda install -c conda-forge jupyterlab
jupyter lab
```

The next cell confirms that the notebook is running inside the QIIME 2 environment.

In [ ]:
%matplotlib inline
import qiime2
print("QIIME 2", qiime2.__version__)

## 3. Load data from Qiita

[Qiita](https://qiita.ucsd.edu) stores the class study: the raw reads, the sample metadata, and the processed feature tables. The instructors process the reads (demultiplexing, Deblur, Greengenes2), so you start from three files. Download them from Qiita to your computer, then put them in the `data/` folder next to this notebook. **In a Codespace**, drag the files onto the `data` folder in the file explorer on the left, or right-click `data` → **Upload...**

| File | What it is | Where to find it in Qiita |
|---|---|---|
| Feature table (`.biom` or `.qza`) | Read counts, samples × ASVs | Study page → data type (16S) → preparation → Deblur feature table → **Download** |
| Sample information (`.tsv`) | One row per sample: body site, participant, sample type, questionnaire | Study page → **Sample Information** → Download |
| Taxonomy (`.qza` or `.tsv`) | Greengenes2 name for each ASV | Distributed with the feature table; if the BIOM table already carries taxonomy, leave `TAXONOMY` empty |

**EDIT** the file names and metadata column names below to match your download.

In [ ]:
# EDIT: paths to the files downloaded from Qiita
TABLE = "data/table.biom"            # .biom or .qza
METADATA = "data/sample_information.tsv"
TAXONOMY = "data/taxonomy.qza"       # .qza or .tsv; set to None if the table carries taxonomy
TREE = None                          # optional rooted tree (.qza) for Faith's PD and UniFrac

# EDIT: metadata columns
GROUP = "body_site"                  # the comparison of interest
SUBJECT = "host_subject_id"          # participant identifier
CONTROL_COL = "sample_type"          # column that marks blanks and mock communities
CONTROL_VALUES = ["control blank", "control mock"]

In [ ]:
import warnings
from pathlib import Path

import biom
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import skbio
from qiime2 import Artifact, Metadata
from qiime2.plugins import composition, diversity, feature_table, taxa

warnings.filterwarnings("ignore", category=FutureWarning)
OUT = Path("results"); OUT.mkdir(exist_ok=True)


def load_table(path):
    if str(path).endswith(".qza"):
        return Artifact.load(path), None
    bt = biom.load_table(path)
    tax = bt.metadata_to_dataframe("observation") if bt.metadata(axis="observation") else None
    return Artifact.import_data("FeatureTable[Frequency]", path, view_type="BIOMV210Format"), tax


def load_taxonomy(path, from_table):
    if path is None:
        if from_table is None:
            raise ValueError("No taxonomy file given and the table carries no taxonomy.")
        cols = [c for c in from_table.columns if c.startswith("taxonomy")]
        taxon = from_table[cols].astype(str).agg("; ".join, axis=1)
        df = pd.DataFrame({"Taxon": taxon})
        df.index.name = "Feature ID"
        return Artifact.import_data("FeatureData[Taxonomy]", df)
    if str(path).endswith(".qza"):
        return Artifact.load(path)
    return Artifact.import_data("FeatureData[Taxonomy]", path, view_type="TSVTaxonomyFormat")


table, tax_from_table = load_table(TABLE)
taxonomy = load_taxonomy(TAXONOMY, tax_from_table)
metadata = Metadata.load(METADATA)
tree = Artifact.load(TREE) if TREE else None

counts = table.view(pd.DataFrame)                 # samples x features
meta = metadata.to_dataframe()
shared = counts.index.intersection(meta.index)
print(f"{counts.shape[0]} samples and {counts.shape[1]} features in the table; "
      f"{len(shared)} samples have metadata")
counts, meta = counts.loc[shared], meta.loc[shared]

depth = counts.sum(axis=1).rename("reads")
is_control = meta[CONTROL_COL].isin(CONTROL_VALUES) if CONTROL_COL in meta else pd.Series(False, index=meta.index)
print(depth.groupby(is_control.map({True: "control", False: "sample"})).describe().round(0))

Before any analysis, look at the controls: blanks should have few reads, and the mock community should contain only its known members. The next cell lists the most abundant features in each control, then removes controls from the dataset used for the rest of the notebook.

In [ ]:
tax_series = taxonomy.view(pd.Series)
for sid in meta.index[is_control]:
    top = counts.loc[sid].sort_values(ascending=False).head(5)
    print(f"\n{sid} ({meta.loc[sid, CONTROL_COL]}, {int(depth[sid])} reads)")
    for fid, n in top[top > 0].items():
        print(f"  {n:>6.0f}  {tax_series.get(fid, fid)}")

keep = meta.index[~is_control]
table = feature_table.methods.filter_samples(table=table, metadata=Metadata(meta.loc[keep])).filtered_table
meta = meta.loc[keep]
md_keep = Metadata(meta)
print(f"\n{len(keep)} biological samples retained")

## 4. Taxonomic composition

Collapse ASVs to genus (level 6 of the Greengenes2 taxonomy), convert to relative abundance, and plot the most abundant genera per sample. The interactive QIIME 2 bar plot is also saved to `results/taxa-barplot.qzv`; open it at [view.qiime2.org](https://view.qiime2.org).

In [ ]:
TOP_N = 12  # EDIT: number of genera to show

genus = taxa.methods.collapse(table=table, taxonomy=taxonomy, level=6).collapsed_table
rel = feature_table.methods.relative_frequency(table=genus).relative_frequency_table.view(pd.DataFrame)
rel.columns = [c.split(";")[-1].strip() or c for c in rel.columns]
rel = rel.T.groupby(level=0).sum().T                      # merge duplicate genus labels

top = rel.mean().sort_values(ascending=False).index[:TOP_N]
plot_df = rel[top].assign(Other=1 - rel[top].sum(axis=1))
order = meta.sort_values([GROUP]).index
plot_df = plot_df.loc[order]

fig, ax = plt.subplots(figsize=(max(8, len(order) * 0.25), 5))
plot_df.plot.bar(stacked=True, width=0.9, ax=ax, colormap="tab20", legend=False)
ax.set_xticklabels([f"{meta.loc[s, GROUP]}" for s in order], rotation=90, fontsize=7)
ax.set_ylabel("Relative abundance"); ax.set_xlabel(GROUP); ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, title="Genus")
plt.tight_layout(); plt.show()

print(rel.groupby(meta[GROUP]).mean()[top].T.round(3))
taxa.visualizers.barplot(table=table, taxonomy=taxonomy, metadata=md_keep).visualization.save(str(OUT / "taxa-barplot.qzv"))

## 5. Rarefaction, alpha diversity, and beta diversity

Samples are sequenced to different depths, and deeper samples appear more diverse simply because more reads were drawn. Rarefaction subsamples every sample, without replacement, to a common depth. Use the curves to choose a depth where most samples have plateaued, and note how many samples fall below it and are lost.

In [ ]:
from skbio.stats import subsample_counts

counts = table.view(pd.DataFrame)
depth = counts.sum(axis=1)
print(depth.describe().round(0))

steps = np.linspace(100, np.percentile(depth, 90), 10).astype(int)
curves = {}
for sid, row in counts.iterrows():
    v = row.values.astype(int)
    curves[sid] = [np.mean([(subsample_counts(v, d, seed=i) > 0).sum() for i in range(3)])
                   if d <= v.sum() else np.nan for d in steps]
curves = pd.DataFrame(curves, index=steps)

fig, ax = plt.subplots(figsize=(7, 4))
for g, ids in meta.groupby(GROUP).groups.items():
    ax.plot(curves.index, curves[ids].mean(axis=1), marker="o", label=str(g))
ax.set_xlabel("Sequencing depth (reads)"); ax.set_ylabel("Observed features (mean)")
ax.legend(title=GROUP); plt.tight_layout(); plt.show()

In [ ]:
DEPTH = 1000  # EDIT: rarefaction depth chosen from the curves above

lost = depth[depth < DEPTH].index
print(f"Rarefying to {DEPTH} reads drops {len(lost)} of {len(depth)} samples: {list(lost)}")

if tree is not None:
    core = diversity.pipelines.core_metrics_phylogenetic(table=table, phylogeny=tree, sampling_depth=DEPTH, metadata=md_keep)
    alpha_names = ["observed_features", "shannon", "faith_pd"]
    beta_names = ["bray_curtis", "jaccard", "unweighted_unifrac", "weighted_unifrac"]
else:
    core = diversity.pipelines.core_metrics(table=table, sampling_depth=DEPTH, metadata=md_keep)
    alpha_names = ["observed_features", "shannon"]
    beta_names = ["bray_curtis", "jaccard"]

alpha = pd.concat({n: getattr(core, f"{n}_vector").view(pd.Series) for n in alpha_names}, axis=1)
alpha = alpha.join(meta[[GROUP]])
print(alpha.groupby(GROUP).agg(["mean", "std"]).round(2))

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, len(alpha_names), figsize=(4 * len(alpha_names), 4))
for ax, m in zip(np.atleast_1d(axes), alpha_names):
    sns.boxplot(data=alpha, x=GROUP, y=m, ax=ax, color="white")
    sns.stripplot(data=alpha, x=GROUP, y=m, ax=ax, color="black", size=4)
    groups = [g[m].values for _, g in alpha.groupby(GROUP)]
    test = stats.mannwhitneyu(*groups) if len(groups) == 2 else stats.kruskal(*groups)
    ax.set_title(f"{m}\n{type(test).__name__.replace('Result', '')} p = {test.pvalue:.3g}", fontsize=10)
plt.tight_layout(); plt.show()

These tests treat every sample as independent. If each participant contributes one sample per group (a paired design), use a paired test instead, for example `scipy.stats.wilcoxon` on per-participant differences.

## 6. Ordination (PCoA) and PERMANOVA

Each beta diversity metric summarizes how different two samples are; PCoA projects the resulting distance matrix into two dimensions. PERMANOVA tests whether samples in different groups are more different from each other than samples within a group. Compare the results across metrics: if they disagree, ask why. Interactive Emperor plots are saved to `results/`.

In [ ]:
from skbio.stats.distance import permanova

rows = []
fig, axes = plt.subplots(1, len(beta_names), figsize=(4.5 * len(beta_names), 4.2))
for ax, m in zip(np.atleast_1d(axes), beta_names):
    dm = getattr(core, f"{m}_distance_matrix").view(skbio.DistanceMatrix)
    pc = getattr(core, f"{m}_pcoa_results").view(skbio.OrdinationResults)
    coords = pc.samples.iloc[:, :2].join(meta[[GROUP]])
    expl = pc.proportion_explained.iloc[:2] * 100
    sns.scatterplot(data=coords, x=coords.columns[0], y=coords.columns[1], hue=GROUP, ax=ax, s=40)
    ax.set_xlabel(f"PC1 ({expl.iloc[0]:.0f}%)"); ax.set_ylabel(f"PC2 ({expl.iloc[1]:.0f}%)"); ax.set_title(m)
    getattr(core, f"{m}_emperor").save(str(OUT / f"{m}-emperor.qzv"))
    for col in [GROUP, SUBJECT]:
        if col in meta and meta.loc[list(dm.ids), col].nunique() > 1:
            r = permanova(dm, meta.loc[list(dm.ids), col], permutations=999)
            rows.append({"metric": m, "grouping": col, "pseudo-F": round(r["test statistic"], 2), "p": r["p-value"]})
plt.tight_layout(); plt.show()
pd.DataFrame(rows)

PERMANOVA is also sensitive to differences in spread (dispersion) between groups, not only to differences in location. A significant result with very unequal group spreads should be checked with `skbio.stats.distance.permdisp`.

## 7. Differential abundance

Which genera differ between groups? Because 16S data are compositional (each sample sums to 1), an increase in one taxon makes every other taxon look smaller, so ordinary per-taxon tests on relative abundances are unreliable. ANCOM-BC models this bias explicitly. Filter out rare genera first, since they cannot be tested meaningfully, and treat any result from a small study as a hypothesis rather than a finding.

In [ ]:
MIN_PREVALENCE = 0.25  # EDIT: keep genera present in at least this fraction of samples

g = genus.view(pd.DataFrame)
min_samples = int(np.ceil(MIN_PREVALENCE * len(g)))
genus_f = feature_table.methods.filter_features(table=genus, min_samples=min_samples).filtered_table
print(f"Testing {genus_f.view(pd.DataFrame).shape[1]} of {g.shape[1]} genera")

res = composition.methods.ancombc(table=genus_f, metadata=md_keep, formula=GROUP)
composition.visualizers.da_barplot(data=res.differentials, significance_threshold=0.05).visualization.save(str(OUT / "ancombc-barplot.qzv"))
res.differentials

Questions to answer before believing a differentially abundant genus:

- How many genera were tested, and is the reported q-value corrected for that number?
- Is the genus abundant enough in the bar plot to be a plausible biological difference, or is it rare?
- Does it also appear in the blanks?
- Would the result survive a second method (for example, `skbio.stats.composition.dirmult_ttest`)?